Using Contracts Finder API to get Company-Level Contract Signals 

This notebook uses Contracts Finder OCDS award releases to:

1. Pull recent public-sector contract award notices.
2. Extract supplier companies using Companies House numbers.
3. Where available, extract buyer companies using Companies House numbers.
4. Transform contract-level rows into company-level rows.
5. Count:
   - contracts received as supplier
   - contracts given as buyer
   - total contract value received
   - total contract value given
6. Merge with Companies House static company-type data.
7. Perform EDA to understand what types of companies are receiving/giving contracts.

Important limitation:
Contracts Finder buyers are usually public sector organisations, not normal Companies House companies. Therefore, buyer side company-level rows may be very limited or zero.
This code is pulling public sector contract award data from Contracts Finder, finding the Companies House number of winning suppliers, summarising contract wins per company

Each award notice has two sides:

- Buyer — the public-sector organisation that issues and pays for the contract
  (e.g. a council, NHS trust, government department). It receives the goods/services.
- Supplier — the company that wins the contract, delivers the work, and gets paid.

In [ ]:
# Import, config and cache helpers
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv


CF_BASE = "https://www.contractsfinder.service.gov.uk"
PROC_DIR  = Path(r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\processed")
CACHE_DIR = Path(r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

#  18-month cutoff date from today. This keeps the analysis focused on recent contract activity rather than old awards.
RECENT_CUTOFF = pd.Timestamp.today().normalize() - pd.DateOffset(months=18)

def load_json_cache(path):
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None
def save_json_cache(path, obj):
    with open(Path(path), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

# Formatting money values so that it is easily readable
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

def format_money(x):
    """Format numeric values as pounds with commas."""
    if pd.isna(x):
        return ""
    return f"£{x:,.0f}"

def format_number(x):
    """Format numeric values with commas and no scientific notation."""
    if pd.isna(x):
        return ""
    return f"{x:,.0f}"


In [68]:
# This block defines the functions used to collect award notices from the Contracts Finder API.
def cf_search(published_from, published_to, stages="award", limit=100, cursor=None):
    """
    Pull one batch of Contracts Finder OCDS notices.
    """
    url = f"{CF_BASE}/Published/Notices/OCDS/Search"

    params = {
        "publishedFrom": published_from,
        "publishedTo": published_to,
        "stages": stages,
        "limit": limit
    }
    if cursor:
        params["cursor"] = cursor
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Pulling multiple batches
def cf_search_all(published_from, published_to, stages="award", limit=100, max_batches=60, pause=1.0):
    """Follow the OCDS links.next cursor to page through all award releases."""

    all_releases = []
    seen_ids = set()
    next_url = None
    batch = 0

    while batch < max_batches:
        try:
            if next_url:
                resp = requests.get(next_url, timeout=30)
            else:
                resp = requests.get(
                    f"{CF_BASE}/Published/Notices/OCDS/Search",
                    params={
                        "publishedFrom": published_from,
                        "publishedTo": published_to,
                        "stages": stages,
                        "limit": limit
                    },
                    timeout=30
                )

            resp.raise_for_status()

        except requests.exceptions.HTTPError as e:
            status = getattr(e.response, "status_code", None)

            if status in (403, 429):
                print(f"Rate limited HTTP {status} on batch {batch}. Waiting 300 seconds.")
                time.sleep(300)
                continue

            raise

        data = resp.json()
        releases = data.get("releases", [])

        if not releases:
            break

        for r in releases:
            rid = r.get("id")

            if rid not in seen_ids:
                seen_ids.add(rid)
                all_releases.append(r)

        next_url = (data.get("links") or {}).get("next")

        batch += 1

        print(f"Batch {batch}: collected {len(all_releases)} unique releases")

        if not next_url:
            break

        time.sleep(pause)

    return all_releases

In [69]:
# generate api results once and reuse from cache
RAW_PATH = CACHE_DIR / "cf_award_releases.json"
def get_award_releases(refresh=False):
    if not refresh:
        cached = load_json_cache(RAW_PATH)
        if cached is not None:
            print(f"Loaded {len(cached)} releases from cache")
            return cached
    releases = cf_search_all(
        published_from=RECENT_CUTOFF.strftime("%Y-%m-%dT00:00:00"),
        published_to=pd.Timestamp.today().strftime("%Y-%m-%dT00:00:00"),
        stages="award",
        limit=100,
        max_batches=60,
    )
    save_json_cache(RAW_PATH, releases)
    print(f"Harvested and cached {len(releases)} releases")
    return releases
award_releases = get_award_releases(refresh=False)
print("Unique releases:", len(award_releases))
print("Unique ocids:", len({r.get('ocid') for r in award_releases}))

Loaded 6000 releases from cache
Unique releases: 6000
Unique ocids: 5715


This means those 6,000 releases relate to 5,715 unique contracting processes. The number of releases is slightly higher than the number of unique ocids because one contracting process can sometimes appear in more than one release.

In [ ]:
#taking one OCDS release and turning it into multiple rows
"""This code block converts the nested Contracts Finder JSON data into a normal table format.

The function flatten_release(release) takes one OCDS release and turns it into multiple rows.

First, it extracts basic information from the release:

ocid: the contracting process ID
tender_title: the contract or tender title
buyer_id: the buyer’s ID
buyer_name: the organisation buying the goods or services

The code then loops through the parties section of the release.
 It looks for parties that have a Companies House identifier using the scheme GB-COH. These identifiers are stored in the dictionary party_ch."""
def flatten_release(release):
    rows = []
    ocid = release.get("ocid")
    tender_title = (release.get("tender") or {}).get("title")
    buyer = release.get("buyer") or {}
    buyer_id = buyer.get("id")
    buyer_name = buyer.get("name")

    # Party id -> Companies House number, only for parties published with a GB-COH identifier.
    party_ch = {}
    for party in release.get("parties", []):
        ident = party.get("identifier") or {}
        if ident.get("scheme") == "GB-COH" and ident.get("id"):
            party_ch[party.get("id")] = str(ident.get("id")).strip().upper().zfill(8)

    # SUPPLIER rows: one per supplier per award (the company that received the contract).
    for award in release.get("awards", []):
        a_date = award.get("date")
        a_value = (award.get("value") or {}).get("amount")
        suppliers = award.get("suppliers", [])
        n_sup = len(suppliers) or 1
        share = (a_value / n_sup) if a_value is not None else None

        for supplier in suppliers:
            cn = party_ch.get(supplier.get("id"))
            if cn is None:
                continue
            rows.append({
                "CompanyNumber": cn,
                "company_name_cf": supplier.get("name"),
                "role": "supplier",
                "counterparty_name": buyer_name,
                "contract_title": tender_title,
                "award_date": a_date,
                "award_value_gbp": a_value,          
                "award_value_share_gbp": share,      
                "num_suppliers_on_award": n_sup,
                "ocid": ocid,
        })

    # BUYER row: only if the buyer itself has a Companies House number (the company that gave the contract).
    buyer_cn = party_ch.get(buyer_id)
    if buyer_cn is not None:
        dates  = [aw.get("date") for aw in release.get("awards", []) if aw.get("date")]
        total  = sum(((aw.get("value") or {}).get("amount") or 0) for aw in release.get("awards", []))
        rows.append({
            "CompanyNumber": buyer_cn,
            "company_name_cf": buyer_name,
            "role": "buyer",
            "counterparty_name": None,
            "contract_title": tender_title,
            "award_date": max(dates) if dates else None,
            "award_value_gbp": total or None,
            "ocid": ocid,
        })
    return rows


raw_rows = []
for rel in award_releases:
    raw_rows.extend(flatten_release(rel))

rows_df = pd.DataFrame(raw_rows)
rows_df["CompanyNumber"] = rows_df["CompanyNumber"].astype(str).str.upper().str.zfill(8)
rows_df["award_date"] = pd.to_datetime(rows_df["award_date"], errors="coerce", utc=True)
rows_df["award_value_gbp"] = pd.to_numeric(rows_df["award_value_gbp"], errors="coerce")

# Applying the recency cutoff
rows_df = rows_df[rows_df["award_date"] >= RECENT_CUTOFF.tz_localize("UTC")]
print("Contract rows after flatten + cutoff:", len(rows_df))
rows_df.head()

Contract rows after flatten + cutoff: 2243


,CompanyNumber,company_name_cf,role,counterparty_name,contract_title,award_date,award_value_gbp,award_value_share_gbp,num_suppliers_on_award,ocid
0,03176761,ANS Group Limited,supplier,MINISTRY OF DEFENCE,Remote Access Movements Portal (RAMP) and WATERGUARD Applications (WG Apps) ...,2026-03-27 00:00:00+00:00,"4,492,097.64","4,492,097.64",1,ocds-b5fd17-ec0dd496-0b36-418b-a6f3-e857038569e9
1,00947968,CGI (EUROPE) LTD,supplier,MINISTRY OF DEFENCE,WATERGUARD Applications (WG Apps) Sustainment 2026-29,2026-04-14 23:00:00+00:00,"3,470,777.39","3,470,777.39",1,ocds-b5fd17-aeb918b5-3b11-4cc9-ba25-8f93d85b960e
2,03498080,Ion Property Developments Limited,supplier,WAKEFIELD COUNCIL,"Old Westgate Station Hotel Development, Wakefield",2026-02-25 00:00:00+00:00,"8,000,000.00","8,000,000.00",1,ocds-b5fd17-9fb7446f-218c-48d6-812b-d41912637d92
3,05510758,Banner Group Ltd,supplier,CPS,Office Supplies and Electronic Office Supplies,2025-05-18 23:00:00+00:00,"802,853.00","802,853.00",1,ocds-b5fd17-cd5303d5-12b9-43f0-b140-ab849243813f
4,OC306448,Browne Jacobson LLP,supplier,HAMPSHIRE COUNTY COUNCIL,External Legal Advice,2026-04-29 23:00:00+00:00,"50,000.00","50,000.00",1,ocds-b5fd17-2a19ab80-1305-4588-af8b-348efbc64b3e


In [ ]:
# making sure that dates and values are in the correct format
rows_df["award_date"] = pd.to_datetime(rows_df["award_date"], errors="coerce", utc=True)
rows_df["award_value_gbp"] = pd.to_numeric(rows_df["award_value_gbp"], errors="coerce")

# Standardise Companies House numbers
rows_df["CompanyNumber"] = (rows_df["CompanyNumber"].astype("string").str.strip().str.upper().str.zfill(8))

# Keeping only recent contracts
rows_df = rows_df[rows_df["award_date"] >= RECENT_CUTOFF.tz_localize("UTC")].copy()

# Removing duplicate supplier contract combinations
rows_df = rows_df.drop_duplicates(["ocid", "CompanyNumber", "role"])

print("Contract-level rows after cleaning:", len(rows_df))
rows_df.head()

Contract-level rows after cleaning: 2154


,CompanyNumber,company_name_cf,role,counterparty_name,contract_title,award_date,award_value_gbp,award_value_share_gbp,num_suppliers_on_award,ocid
0,03176761,ANS Group Limited,supplier,MINISTRY OF DEFENCE,Remote Access Movements Portal (RAMP) and WATERGUARD Applications (WG Apps) ...,2026-03-27 00:00:00+00:00,"4,492,097.64","4,492,097.64",1,ocds-b5fd17-ec0dd496-0b36-418b-a6f3-e857038569e9
1,00947968,CGI (EUROPE) LTD,supplier,MINISTRY OF DEFENCE,WATERGUARD Applications (WG Apps) Sustainment 2026-29,2026-04-14 23:00:00+00:00,"3,470,777.39","3,470,777.39",1,ocds-b5fd17-aeb918b5-3b11-4cc9-ba25-8f93d85b960e
2,03498080,Ion Property Developments Limited,supplier,WAKEFIELD COUNCIL,"Old Westgate Station Hotel Development, Wakefield",2026-02-25 00:00:00+00:00,"8,000,000.00","8,000,000.00",1,ocds-b5fd17-9fb7446f-218c-48d6-812b-d41912637d92
3,05510758,Banner Group Ltd,supplier,CPS,Office Supplies and Electronic Office Supplies,2025-05-18 23:00:00+00:00,"802,853.00","802,853.00",1,ocds-b5fd17-cd5303d5-12b9-43f0-b140-ab849243813f
4,OC306448,Browne Jacobson LLP,supplier,HAMPSHIRE COUNTY COUNCIL,External Legal Advice,2026-04-29 23:00:00+00:00,"50,000.00","50,000.00",1,ocds-b5fd17-2a19ab80-1305-4588-af8b-348efbc64b3e


## From one row per contract to one row per company

Up to this point `rows_df` is at **contract grain**: each row is a single
supplier-on-an-award or a single buyer-on-a-release. A company that won five
contracts appears in five rows. For the Relationship Manager (RM) view we want
**company grain**: one row per company, with its contract activity rolled up.

| Column | Meaning |
|---|---|
| `contracts_won_as_supplier` | number of distinct awards the company won (supplier) |
| `total_value_won_gbp` | total £ value of those winning awards |
| `contracts_given_as_buyer` | number of distinct awards the company gave out (buyer) |
| `total_value_given_gbp` | total £ value of those awarded-out contracts |
| `total_contracts_linked` | supplier + buyer count (all activity touching this company) |
| `total_value_linked_gbp` | supplier + buyer value |

We count distinct **`ocid`** (the OCDS contracting-process id), not rows, so a
process that lists the same company twice is still counted once. [1]

In [79]:
# Split the contract-level rows by role, then aggregate each side to company grain.
supplier_rows = rows_df[rows_df["role"] == "supplier"].copy()
buyer_rows = rows_df[rows_df["role"] == "buyer"].copy()

# Supplier side: the company won/received the contract 
supplier_summary = (
    supplier_rows
    .groupby("CompanyNumber", as_index=False)
    .agg(
        company_name=("company_name_cf", "first"),
        contracts_won_as_supplier=("ocid", "nunique"),
        total_value_won_gbp=("award_value_gbp", "sum"),
        avg_value_won_gbp=("award_value_gbp", "mean"),
        latest_supplier_award_date=("award_date", "max"),
        top_buyers=("counterparty_name",
                    lambda s: "; ".join(sorted(set(s.dropna().astype(str)))[:5])),
    )
)

# Buyer side: the company gave out/awarded the contract
# In practice most buyers are councils / NHS trusts with no Companies House
buyer_summary = (
    buyer_rows
    .groupby("CompanyNumber", as_index=False)
    .agg(
        company_name_buyer=("company_name_cf", "first"),
        contracts_given_as_buyer=("ocid", "nunique"),
        total_value_given_gbp=("award_value_gbp", "sum"),
        latest_award_given_date=("award_date", "max"),
    )
)

# combining both sides into one row per company 
company_contracts = supplier_summary.merge(buyer_summary, on="CompanyNumber", how="outer")

# A company that only appears as a buyer has no supplier-side name yet: coalesce.
company_contracts["company_name"] = (
    company_contracts["company_name"].fillna(company_contracts["company_name_buyer"])
)
company_contracts = company_contracts.drop(columns=["company_name_buyer"])

# Filling the gaps left by the outer merge (a company on only one side).
count_cols = ["contracts_won_as_supplier", "contracts_given_as_buyer"]
value_cols = ["total_value_won_gbp", "avg_value_won_gbp", "total_value_given_gbp"]
company_contracts[count_cols] = company_contracts[count_cols].fillna(0).astype(int)
company_contracts[value_cols] = company_contracts[value_cols].fillna(0.0)

# Combined activity across both roles.
company_contracts["total_contracts_linked"] = (
    company_contracts["contracts_won_as_supplier"]
    + company_contracts["contracts_given_as_buyer"]
)
company_contracts["total_value_linked_gbp"] = (
    company_contracts["total_value_won_gbp"]
    + company_contracts["total_value_given_gbp"]
)

company_contracts = (
    company_contracts
    .sort_values("total_value_won_gbp", ascending=False)
    .reset_index(drop=True)
)

print("Contract-level rows in :", len(rows_df))
print("Company-level rows out :", len(company_contracts))
print(" appearing as supplier:", (company_contracts["contracts_won_as_supplier"] > 0).sum())
print(" appearing as buyer   :", (company_contracts["contracts_given_as_buyer"] > 0).sum())
company_contracts.head(20)

Contract-level rows in : 2154
Company-level rows out : 1317
 appearing as supplier: 1317
 appearing as buyer   : 0


,CompanyNumber,company_name,contracts_won_as_supplier,total_value_won_gbp,avg_value_won_gbp,latest_supplier_award_date,top_buyers,contracts_given_as_buyer,total_value_given_gbp,latest_award_given_date,total_contracts_linked,total_value_linked_gbp
0,02722343,Hologic Ltd,2,"543,024,000.00","271,512,000.00",2026-06-14 23:00:00+00:00,Partners Procurement Service; Supply Chain Coordination Limited,0,0.00,NaT,2,"543,024,000.00"
1,03535936,Conmed UK Ltd,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
2,14771423,Lexington Medical UK Ltd,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
3,00966736,KeyMed (Medical & Industrial Equipment) Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
4,06041206,Healthium Medtech UK Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
5,02296559,B.Braun Medical Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
6,03238147,Medline Industries Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
7,00524279,Purple Surgical UK Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
8,01013256,Starkstrom Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"
9,00047888,Uniphar Medtech Limited,1,"543,000,000.00","543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited,0,0.00,NaT,1,"543,000,000.00"


> The buyer side is nearly empty because Contracts Finder only carries a
> Companies House number for a party when the buyer supplied one, and this
> essentially only happens for awarded suppliers — public bodies (councils,
> NHS trusts, central government) are identified by their own government scheme,
> not a `GB-COH` number. [2] So `contracts_given_as_buyer` will be ~0 for almost
> every company. 

## Exploratory analysis: who receives and who gives out contracts?

The questions we want the data to answer:

1. **Coverage** - how much did we actually pull and how many distinct companies?
2. **Value shape** - are these mostly small awards or a few very large ones?
3. **Concentration** -is contract value spread across many SMEs, or captured by a handful of big winners?
4. **Top receivers** - which companies win the most (by count and by value)?
5. **Top givers** - which public bodies hand out the most work?
6. **Type of company** - once we join back to the Companies House base file,
   which *sectors* and *size segments* are winning the work?

### 1. Coverage and role split

In [81]:
role_split = (
    rows_df["role"].value_counts()
    .rename_axis("role").reset_index(name="contract_rows")
)
print("Role split (contract-level rows)")
display(role_split)

print("\nSupplier-side coverage")
print("  Supplier contract rows   :", len(supplier_rows))
print("  Unique supplier companies:", supplier_rows["CompanyNumber"].nunique())
print("  Unique award processes   :", supplier_rows["ocid"].nunique())
print("  Date range               :",
      supplier_rows["award_date"].min(), "->", supplier_rows["award_date"].max())

missing_value = supplier_rows["award_value_gbp"].isna().mean()
print(f"  Awards with no £ value   : {missing_value:.1%}")

Role split (contract-level rows)


,role,contract_rows
0,supplier,2154



Supplier-side coverage
  Supplier contract rows   : 2154
  Unique supplier companies: 1317
  Unique award processes   : 1958
  Date range               : 2025-01-10 00:00:00+00:00 -> 2026-12-05 00:00:00+00:00
  Awards with no £ value   : 0.0%


 The role split confirms the dataset is overwhelmingly
*supplier* rows - companies receiving work, with buyer rows a thin sliver
(only where a buyer happened to carry a company number). The share of awards with
a missing £ value matters for every value-based statistic below: those awards
still count toward how many contracts a company won, but contribute £0 to
value totals, so a company can look busy yet low-value simply because its
awards were published without amounts.

### 2. The shape of contract value

In [82]:
print("Supplier award value (£) — summary")
display(supplier_rows["award_value_gbp"].describe().to_frame("award_value_gbp"))

# Banding 
supplier_rows["contract_value_band"] = pd.cut(
    supplier_rows["award_value_gbp"],
    bins=[0, 10_000, 100_000, 1_000_000, 10_000_000, np.inf],
    labels=["<10k", "10k-100k", "100k-1m", "1m-10m", ">10m"],
)

value_bands = (
    supplier_rows["contract_value_band"].value_counts().sort_index()
    .rename_axis("contract_value_band").reset_index(name="contract_count")
)
value_bands["share_of_awards"] = (
    value_bands["contract_count"] / value_bands["contract_count"].sum()
).round(3)
print("\nContract value bands")
display(value_bands)

Supplier award value (£) — summary


,award_value_gbp
count,"2,154.00"
mean,"17,120,439.54"
std,"88,691,691.99"
min,0.00
25%,"53,576.25"
50%,"148,179.50"
75%,"727,157.25"
max,"543,000,000.00"



Contract value bands


,contract_value_band,contract_count,share_of_awards
0,<10k,14,0.01
1,10k-100k,870,0.41
2,100k-1m,761,0.36
3,1m-10m,304,0.14
4,>10m,169,0.08


 Public-procurement value is heavily right-skewed: the
median award is far below the mean, because a small number of very large
contracts drag the average upward. The banding usually shows the bulk of awards
sitting in the sub-£100k range (typical SME-scale work) with a long thin tail
above £1m. That tail is where a handful of large winners live, which is exactly
what the concentration check next quantifies.

### 3. Concentration — spread out, or captured by a few?

In [ ]:
wins = company_contracts[company_contracts["contracts_won_as_supplier"] > 0].copy()

# 3a. Repeat winners vs one off winners.
one_off = (wins["contracts_won_as_supplier"] == 1).sum()
repeat = (wins["contracts_won_as_supplier"] > 1).sum()
print("Winning companies      :", len(wins))
print("  won exactly 1 contract:", one_off, f"({one_off/len(wins):.1%})")
print("  won 2+ contracts      :", repeat,  f"({repeat/len(wins):.1%})")

# 3b. How much of total value sits with the top winners?
wins_by_value = wins.sort_values("total_value_won_gbp", ascending=False)
total_value = wins_by_value["total_value_won_gbp"].sum()
for n in (10, 50, 100):
    if len(wins_by_value) >= n and total_value > 0:
        share = wins_by_value["total_value_won_gbp"].head(n).sum() / total_value
        print(f"  top {n:>3} companies hold {share:.1%} of total won value")

# 3c. Distribution of contracts won per company.
win_counts = (
    wins["contracts_won_as_supplier"].value_counts().sort_index()
    .rename_axis("contracts_won").reset_index(name="num_companies")
)
print("\nContracts won per company")
display(win_counts.head(15))

Winning companies      : 1317
  won exactly 1 contract: 1030 (78.2%)
  won 2+ contracts      : 287 (21.8%)
  top  10 companies hold 14.7% of total won value
  top  50 companies hold 73.6% of total won value
  top 100 companies hold 94.4% of total won value

Contracts won per company


,contracts_won,num_companies
0,1,1030
1,2,150
2,3,56
3,4,24
4,5,16
5,6,8
6,7,6
7,8,3
8,9,4
9,10,7


Two levers of concentration: breadth (most companies
win just once - a long tail of occasional suppliers) and value capture (a small
top slice holding a large share of total £).Proposition the is the
key strategic read: the one-off, mid-value winners are the interesting
prospects, a company that has just landed its first public contract often needs
working capital or invoice finance to deliver it and is less likely to be
already locked in with a competitor than a mega winner.

### 4. Top companies receiving contracts

In [ ]:
top_by_value = (
    company_contracts
    .sort_values("total_value_won_gbp", ascending=False)
    [["CompanyNumber", "company_name", "contracts_won_as_supplier","total_value_won_gbp", "latest_supplier_award_date", "top_buyers"]].head(20)
)
print("Top 20 companies by total value won")
display(top_by_value)

top_by_count = (
    company_contracts
    .sort_values("contracts_won_as_supplier", ascending=False)
    [["CompanyNumber", "company_name", "contracts_won_as_supplier","total_value_won_gbp", "latest_supplier_award_date"]].head(20)
)
print("\nTop 20 companies by number of contracts won")
display(top_by_count)

Top 20 companies by total value won


,CompanyNumber,company_name,contracts_won_as_supplier,total_value_won_gbp,latest_supplier_award_date,top_buyers
0,02722343,Hologic Ltd,2,"543,024,000.00",2026-06-14 23:00:00+00:00,Partners Procurement Service; Supply Chain Coordination Limited
44,02420424,RB Medical Engineering Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
32,00856254,W.L. Gore and Associates (U.K.) Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
33,05453655,Kebomed UK Ltd.,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
34,09241417,Harris-Jones Medical Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
35,08853263,Schultz Medical (UK) Ltd,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
36,11080821,Biospectrum Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
37,14407269,Winners First Trading Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
38,03090951,Surgical Holdings Limited,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited
39,12992754,Genmed Enterprises International Ltd,1,"543,000,000.00",2026-06-14 23:00:00+00:00,Supply Chain Coordination Limited



Top 20 companies by number of contracts won


,CompanyNumber,company_name,contracts_won_as_supplier,total_value_won_gbp,latest_supplier_award_date
84,02174990,Softcat Plc,44,"35,688,293.44",2026-06-24 23:00:00+00:00
311,03571286,CARE LINE HOMECARE LIMITED,35,"2,635,260.16",2026-08-31 23:00:00+00:00
335,03742352,SPRINGFIELD HOME CARE SERVICES LIMITED,33,"2,026,365.12",2026-06-30 23:00:00+00:00
77,02548628,PHOENIX SOFTWARE LIMITED,30,"52,119,655.48",2026-06-28 23:00:00+00:00
167,02465350,CDW Limited,17,"10,863,770.35",2026-06-23 23:00:00+00:00
379,04430310,Permanent Futures Limited,15,"1,435,034.50",2026-06-18 23:00:00+00:00
391,01102211,Derek Lewis Ltd,15,"1,331,944.26",2026-03-03 00:00:00+00:00
270,01628868,Civica UK Limited,13,"3,804,417.74",2026-06-29 23:00:00+00:00
155,00144585,East Kent Road Car Company Limited,12,"12,693,029.00",2026-03-30 23:00:00+00:00
430,07890835,MATCHTECH GROUP (HOLDINGS) LIMITED,12,"953,368.00",2026-04-12 23:00:00+00:00


### 5. Top organisations giving out contracts

In [84]:
# The "giver" is the buyer. We read it off the supplier rows' counterparty_name,
# which names the public body even when it has no Companies House number.
top_givers = (
    supplier_rows
    .groupby("counterparty_name", as_index=False)
    .agg(
        contracts_given=("ocid", "nunique"),
        total_value_given_gbp=("award_value_gbp", "sum"),
        unique_suppliers_used=("CompanyNumber", "nunique"),
    )
    .sort_values("contracts_given", ascending=False)
    .head(20)
)
print("Top 20 buyers by number of contracts awarded")
display(top_givers)

Top 20 buyers by number of contracts awarded


,counterparty_name,contracts_given,total_value_given_gbp,unique_suppliers_used
54,Darlington Borough Council,78,"5,509,566.40",6
158,Ministry of Justice,65,"126,912,605.17",42
60,Department for Education,61,"204,403,958.79",36
123,Kirklees Council,59,"625,046,135.36",73
295,UK SHARED BUSINESS SERVICES LIMITED,53,"8,366,402.34",31
71,East Sussex County Council,51,"26,759,930.32",12
48,DERBY CITY COUNCIL,41,"6,485,575.46",36
99,HAMPSHIRE COUNTY COUNCIL,35,"4,215,768.03",21
96,H M REVENUE & CUSTOMS,34,"324,688,983.37",29
156,Ministry of Defence,33,"778,285,418.02",28


The contract givers are almost entirely public bodies — councils,
NHS trusts, universities, central-government departments. `unique_suppliers_used`
hints at each buyer's supplier network: a body that spreads work across many
distinct SMEs is a richer source of prospect names than one that routes
everything through a single framework supplier.

### 6. What type of company wins the work? (sector & segment)

In [85]:
# Join company-level contract activity back to the Companies House base file
# (sector / segment / category) on the shared CompanyNumber key. 
# if the file isn't present in this environment, we just skip this block.
BASE_CSV = PROC_DIR / "filtered_bb_sme_sectors.csv"

try:
    wanted = {"CompanyNumber", "CompanyName", "CompanyCategory", "sector", "segment"}
    base_types = pd.read_csv(
        BASE_CSV,
        usecols=lambda c: c.strip() in wanted,   # match on the *stripped* header
        dtype=str,
    )
    base_types.columns = base_types.columns.str.strip()   # clean the kept names

    # defensive: only keep the ones that actually turned up
    keep = [c for c in ["CompanyNumber", "CompanyName", "CompanyCategory", "sector", "segment"]
            if c in base_types.columns]
    base_types = base_types[keep]

    base_types["CompanyNumber"] = (
        base_types["CompanyNumber"].astype(str).str.strip().str.upper().str.zfill(8)
    )
    base_types = base_types.drop_duplicates("CompanyNumber")

    typed = company_contracts.merge(base_types, on="CompanyNumber", how="left")
    matched = typed["sector"].notna().mean()
    print(f"Matched to Companies House base file: {matched:.1%} of contract-winning companies")

    print("\nContract value won by sector")
    by_sector = (
        typed.groupby("sector", dropna=False)
        .agg(companies=("CompanyNumber", "nunique"),
             contracts_won=("contracts_won_as_supplier", "sum"),
             total_value_won_gbp=("total_value_won_gbp", "sum"))
        .sort_values("total_value_won_gbp", ascending=False)
        .reset_index()
    )
    display(by_sector)

    print("\nContract wins by size segment")
    by_segment = (
        typed.groupby("segment", dropna=False)
        .agg(companies=("CompanyNumber", "nunique"),
             contracts_won=("contracts_won_as_supplier", "sum"),
             total_value_won_gbp=("total_value_won_gbp", "sum"))
        .sort_values("total_value_won_gbp", ascending=False)
        .reset_index()
    )
    display(by_segment)

except FileNotFoundError:
    print(f"Base sector file not found at {BASE_CSV} - skipping sector/segment enrichment.")
    print("(The contract-level EDA above stands on its own without it.)")

Matched to Companies House base file: 43.0% of contract-winning companies

Contract value won by sector


,sector,companies,contracts_won,total_value_won_gbp
0,NaN,751,1203,"21,270,430,707.56"
1,Manufacturing,92,131,"12,767,827,831.68"
2,"Technology, legal & professional",257,407,"1,634,828,776.77"
3,Fast growth & emerging,217,413,"1,204,339,457.25"



Contract wins by size segment


,segment,companies,contracts_won,total_value_won_gbp
0,NaN,751,1203,"21,270,430,707.56"
1,Large,290,612,"8,824,771,935.90"
2,Small,195,230,"5,012,268,006.63"
3,Micro,30,38,"1,091,038,079.74"
4,Subsidiary,27,38,"580,350,634.32"
5,Medium,15,23,"92,184,593.11"
6,No Filings,5,5,"3,315,033.00"
7,Dormant,4,5,"3,067,783.00"
